# Assignment 04 — 02: Preference Fine-Tuning (DPO, 5 trials)
**Track 1 / Option A** | Starts from the best SFT model from notebook 01.

Group: **Abdullah Iqbal (26904), Anushe Ali (26418)**

Merges the best SFT LoRA adapter into the base model, then runs 5 DPO+LoRA trials
(varying beta / LR / batch / epochs) and selects the best (tie-break = val loss).

> **Platform: Kaggle (T4).** 00 and 01 were run on Colab. Copy these from your
> Drive into a **Kaggle Dataset** and attach it as input (right panel → *+ Add Input*):
> `test_set.json`, `results/sft_trials.json`, and the best SFT adapter folder
> (e.g. `adapters/sft_trial3/`). Enable **Internet** and a **GPU** in Settings.

## 1. Install & mount

In [ ]:
# Kaggle: turn ON Internet and GPU in the right-hand Settings panel first.
# Run once, then restart the kernel if prompted (Run -> Restart & clear cell outputs).
# Remove preinstalled torchao that upgraded transformers rejects on import
# (we don't use it; -y is a harmless no-op if it isn't installed).
!pip uninstall -y torchao
!pip install -q -U "transformers>=4.51" "trl>=0.15" "peft>=0.11" \
    "datasets>=2.19" accelerate sacrebleu bert-score matplotlib

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, whi

In [ ]:
import os, glob
# --- Kaggle setup ---------------------------------------------------------
# Pin to ONE GPU (Kaggle gives 2x T4); stops accelerate from sharding the model
# across both during training. Must be set before CUDA initializes.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Writable output dir (saved as this notebook's /kaggle/working output).
PROJ = '/kaggle/working/assignment-4'
os.makedirs(PROJ + '/results', exist_ok=True)
os.makedirs(PROJ + '/adapters', exist_ok=True)

# Inputs are split across two attached datasets ('my-data', 'adapters') with
# timestamped subfolders, so locate each file by name ANYWHERE under /kaggle/input.
def _find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    return hits[0] if hits else None

TEST_PATH     = _find('test_set.json')
SFT_META_PATH = _find('sft_trials.json')
assert TEST_PATH,     'test_set.json not found under /kaggle/input - attach the data dataset.'
assert SFT_META_PATH, 'sft_trials.json not found under /kaggle/input - attach the results dataset.'
print('test_set.json   :', TEST_PATH)
print('sft_trials.json :', SFT_META_PATH)
print('Saving outputs to:', PROJ)

test_set.json   : /kaggle/input/datasets/abdullahiqbaldev/my-data/data-20260531T015403Z-3-001/data/test_set.json
sft_trials.json : /kaggle/input/datasets/abdullahiqbaldev/my-data/results-20260531T015329Z-3-001/results/sft_trials.json
Saving outputs to: /kaggle/working/assignment-4


In [ ]:
import json, torch
PROMPT_TEMPLATE = '### Instruction:\n{instruction}\n\n### Response:\n'
def format_prompt(instruction):
    return PROMPT_TEMPLATE.format(instruction=instruction.strip())
def pick_dtype():
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32

@torch.no_grad()
def generate_response(model, tokenizer, instruction, max_new_tokens=256):
    prompt = format_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def generate_all(model, tokenizer, test_set, max_new_tokens=256):
    rows = []
    for ex in test_set:
        rows.append({'id': ex['id'], 'instruction': ex['instruction'],
                     'reference': ex['reference'],
                     'response': generate_response(model, tokenizer, ex['instruction'], max_new_tokens)})
    return rows

def compute_bleu(hyps, refs):
    import sacrebleu
    s = [sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(hyps, refs)]
    return sum(s) / max(len(s), 1)

def compute_bertscore(hyps, refs, model_type='roberta-large'):
    from bert_score import score as bert_score
    P, R, F1 = bert_score(hyps, refs, lang='en', model_type=model_type, verbose=False)
    return float(F1.mean())

def evaluate_rows(rows, bertscore_model='roberta-large'):
    hyps = [r['response'] for r in rows]; refs = [r['reference'] for r in rows]
    bleu = compute_bleu(hyps, refs); bert = compute_bertscore(hyps, refs, bertscore_model)
    return {'bleu': bleu, 'bertscore_f1': bert, 'composite': 0.5*(bleu/100.0)+0.5*bert}

def _vl(t):
    v = t.get('val_loss');  return float('inf') if v is None else v
def select_best(trials, tol=0.005):
    ranked = sorted(trials, key=lambda t: t['composite'], reverse=True)
    top = ranked[0]['composite']
    cont = [t for t in ranked if top - t['composite'] <= tol]
    if len(cont) > 1:
        cont = sorted(cont, key=_vl)
    return cont[0]

In [ ]:
import json
with open(TEST_PATH) as f:
    test_set = json.load(f)
assert all('<<PASTE' not in ex['reference'] for ex in test_set), \
    'Fill in the gold reference answers in test_set.json before running!'
print(len(test_set), 'test prompts loaded')

10 test prompts loaded


## 2. Build the SFT starting point
Load base, apply the best SFT adapter, and `merge_and_unload()` so DPO starts from
the instruction-tuned weights. A fresh LoRA is then trained on top during DPO.

In [ ]:
MODEL_ID = 'Qwen/Qwen3-0.6B-Base'
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os, json, glob

# sft_trials.json was written on Colab, so 'best_adapter' is a Colab path
# (/content/drive/...). Take just the folder NAME and find it under /kaggle/input.
sft_meta = json.load(open(SFT_META_PATH))
best_name = os.path.basename(sft_meta['best_adapter'].rstrip('/'))   # e.g. 'sft_trial3'
_cfgs = sorted(glob.glob(f'/kaggle/input/**/{best_name}/adapter_config.json', recursive=True))
assert _cfgs, (f"Best SFT adapter '{best_name}' not found under /kaggle/input. "
               f"Make sure the adapters dataset is attached.")
best_sft = os.path.dirname(_cfgs[0])
print('Best SFT trial:', sft_meta['best_trial'], '| adapter:', best_sft)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
dtype = pick_dtype()
def load_sft_model():
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map='auto')
    return PeftModel.from_pretrained(base, best_sft).merge_and_unload()
print('SFT loader ready.')

Best SFT trial: 3 | adapter: /kaggle/input/datasets/abdullahiqbaldev/adapters/adapters/sft_trial3


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

SFT loader ready.


## 3. Load and format the preference dataset
Dataset: `trl-lib/ultrafeedback_binarized` (chosen / rejected pairs).
We format the prompt with the same template so it matches SFT.

In [ ]:
from datasets import load_dataset
N_PREF = 2000   # justify subset in report
pref = load_dataset('trl-lib/ultrafeedback_binarized', split='train')
pref = pref.shuffle(seed=42).select(range(min(N_PREF, len(pref))))
def fmt(ex):
    # chosen/rejected are chat lists; take the assistant turn as text.
    def turn(msgs):
        return msgs[-1]['content'] if isinstance(msgs, list) else msgs
    user = ex['chosen'][0]['content'] if isinstance(ex['chosen'], list) else ex['prompt']
    return {'prompt': format_prompt(user),
            'chosen': turn(ex['chosen']), 'rejected': turn(ex['rejected'])}
pref = pref.map(fmt, remove_columns=pref.column_names)
pref = pref.train_test_split(test_size=0.1, seed=42)
ptrain, pval = pref['train'], pref['test']
print(ptrain, '\n', ptrain[0])

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 1800
}) 
 {'chosen': "Sure! Here is a feedback summary for the candidate based on their answer to the question about Have Backbone, Disagree & Commit:\n\nOverall, the candidate demonstrated the ability to handle a difficult situation where they strongly disagreed with their manager's approach to a project. They showed courage in expressing their concerns and reasons for their disagreement, and they tried to explain their perspective to their manager. However, they could have done a few things differently to handle the situation more effectively.\n\nFirstly, the candidate should have been more assertive in their communication with their manager. Instead of just trying to explain their perspective, they should have made a clear and confident case for their approach, highlighting the potential risks and benefits. They could have used data and examples to support their argument, and they should have been more persuasiv

## 4. Define the 5 DPO trials

In [ ]:
# 5 DPO trials varying beta, LR, effective batch, epochs.
# DPO runs TWO forward passes (policy + reference), so per-device batch is kept at 2
# on the T4; effective batch = batch * grad_accum.
DPO_TRIALS = [
    dict(trial=3, beta=0.05, lr=5e-5, batch=2, grad_accum=2, epochs=1),   # eff 4
    dict(trial=4, beta=0.3,  lr=5e-5, batch=2, grad_accum=2, epochs=2),   # eff 4
    dict(trial=5, beta=0.1,  lr=2e-5, batch=2, grad_accum=4, epochs=1),   # eff 8
]

## 5. DPO training loop

In [ ]:
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig
import gc, json, os, traceback
use_bf16 = dtype == torch.bfloat16

dpo_results, skipped = [], []
for cfg in DPO_TRIALS:
    print('\n==== DPO TRIAL', cfg['trial'], cfg, '====')
    out_dir = PROJ + f"/adapters/dpo_trial{cfg['trial']}"
    res_path = PROJ + f"/results/dpo_trial{cfg['trial']}.json"
    if os.path.exists(res_path):
        dpo_results.append(json.load(open(res_path)))
        print('  [resume] loaded saved result, skipping.'); continue
    model = trainer = m = None
    try:
        if os.path.isdir(out_dir) and os.path.exists(out_dir + '/adapter_config.json'):
            print('  [resume] adapter found, evaluating without retraining.')
            m = PeftModel.from_pretrained(load_sft_model(), out_dir)
            m.config.use_cache = True; m.eval()
            rows = generate_all(m, tokenizer, test_set); metrics = evaluate_rows(rows)
            rec = dict(trial=cfg['trial'], **metrics, val_loss=None, config=dict(cfg),
                       rows=rows, adapter_dir=out_dir)
        else:
            model = load_sft_model()
            model.config.use_cache = False
            lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                              target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type='CAUSAL_LM')
            args = DPOConfig(output_dir=out_dir, beta=cfg['beta'], learning_rate=cfg['lr'],
                num_train_epochs=cfg['epochs'], per_device_train_batch_size=cfg['batch'],
                per_device_eval_batch_size=2, gradient_accumulation_steps=cfg['grad_accum'],
                eval_strategy='epoch', save_strategy='no', logging_steps=25,
                max_length=512, report_to='none',
                gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
                bf16=use_bf16, fp16=not use_bf16)
            trainer = DPOTrainer(model=model, args=args, train_dataset=ptrain,
                                 eval_dataset=pval, peft_config=lora, processing_class=tokenizer)
            trainer.train()
            val_loss = trainer.evaluate()['eval_loss']
            trainer.save_model(out_dir)
            model.gradient_checkpointing_disable()
            model.config.use_cache = True; model.eval()
            rows = generate_all(model, tokenizer, test_set); metrics = evaluate_rows(rows)
            rec = dict(trial=cfg['trial'], **metrics, val_loss=val_loss,
                       config=dict(cfg), rows=rows, adapter_dir=out_dir)
        json.dump(rec, open(res_path, 'w'), indent=2)   # checkpoint this trial immediately
        dpo_results.append(rec)
        vl = rec['val_loss'];  vl_s = ('%.4f' % vl) if vl is not None else 'n/a'
        print('Trial %d  BLEU=%.2f  BERT=%.4f  composite=%.4f  val_loss=%s' %
              (cfg['trial'], metrics['bleu'], metrics['bertscore_f1'], metrics['composite'], vl_s))
    except Exception as e:
        # Never let one trial abort a background (Save Version) run: log it, free
        # memory, keep going. Finished trials are already saved to /kaggle/working.
        print(f'  [SKIPPED] trial {cfg["trial"]} failed: {type(e).__name__}: {e}')
        traceback.print_exc(); skipped.append(cfg['trial'])
    finally:
        del model, trainer, m
        gc.collect(); torch.cuda.empty_cache()

if skipped:
    print('\nWARNING: skipped trials (re-run later):', skipped)


==== DPO TRIAL 3 {'trial': 3, 'beta': 0.05, 'lr': 5e-05, 'batch': 2, 'grad_accum': 2, 'epochs': 1} ====


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.665362,0.646279,1.341347,1223950.000000,-0.613425,-0.351027,0.667262,0.146146,0.000198,0.585000,0.145947,-269.227166,-280.703044


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.665362,0.646279,1,1.341347,1223950.000000,-0.613425,-0.351027,0.667262,0.146146,0.000198,0.585000,0.145947,-269.227166,-280.703044


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trial 5  BLEU=11.61  BERT=0.8893  composite=0.5027  val_loss=0.6463


## 6. Results table + best-model selection

In [ ]:
import pandas as pd
df = pd.DataFrame([{k: r[k] for k in ['trial','bleu','bertscore_f1','composite','val_loss']}
                   for r in dpo_results])
display(df)
best = select_best(dpo_results)
print('BEST DPO TRIAL =', best['trial'], '| config:', best['config'])
with open(PROJ + '/results/dpo_trials.json', 'w') as f:
    json.dump({'trials': dpo_results, 'best_trial': best['trial'],
               'best_adapter': best['adapter_dir']}, f, indent=2)
print('Saved dpo_trials.json')

,trial,bleu,bertscore_f1,composite,val_loss
0,3,11.937368,0.886262,0.502818,0.638905
1,4,10.739536,0.889067,0.498231,0.855347
2,5,11.613886,0.889329,0.502734,0.646279


BEST DPO TRIAL = 3 | config: {'trial': 3, 'beta': 0.05, 'lr': 5e-05, 'batch': 2, 'grad_accum': 2, 'epochs': 1}
Saved dpo_trials.json
